# ⚠️ WHO Air Quality Guidelines — Exceedance Analysis

The **WHO 2021 air quality guidelines** define safe exposure limits:

| Pollutant | WHO annual mean | WHO 24-h mean |
|-----------|----------------|---------------|
| NO₂ | 10 µg/m³ | 25 µg/m³ |
| PM2.5 | 5 µg/m³ | 15 µg/m³ |

In this notebook we:
1. Pull daily data for Paris across 2022–2023
2. Flag every day that exceeds the 24-hour guideline
3. Visualise a calendar heatmap of exceedance days

**Requirements**
```
pip install jiskta matplotlib seaborn pandas numpy
```

In [ ]:
# Install the SDK from the private GitHub repo.
# !pip install "jiskta[pandas]" matplotlib


In [ ]:
import os
from jiskta import JisktaClient
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

API_KEY = os.environ.get("JISKTA_API_KEY", "sk_live_YOUR_API_KEY")
client = JisktaClient(api_key=API_KEY)

# WHO 24-hour guideline values (µg/m³)
WHO = {"no2": 25.0, "pm2p5": 15.0}

## 1 — Fetch two years of daily data

In [ ]:
df = client.query(
    lat=(48.7, 49.0),
    lon=(2.2, 2.5),
    start="2022-01",
    end="2023-12",
    variables=["no2", "pm2p5"],
    aggregate="daily",
)

df["date"] = pd.to_datetime(df["date"])
daily = df.groupby("date")[["no2_mean", "pm2p5_mean"]].mean().reset_index()

daily["no2_exceed"]   = daily["no2_mean"]   > WHO["no2"]
daily["pm25_exceed"]  = daily["pm2p5_mean"] > WHO["pm2p5"]
daily["either_exceed"] = daily["no2_exceed"] | daily["pm25_exceed"]

print(f"Days with NO₂  > {WHO['no2']} µg/m³ : {daily['no2_exceed'].sum()}  "
      f"({daily['no2_exceed'].mean()*100:.1f} %)")
print(f"Days with PM2.5 > {WHO['pm2p5']} µg/m³ : {daily['pm25_exceed'].sum()}  "
      f"({daily['pm25_exceed'].mean()*100:.1f} %)")

## 2 — Monthly exceedance bar chart

In [ ]:
monthly = daily.copy()
monthly["year_month"] = monthly["date"].dt.to_period("M")
exc_monthly = monthly.groupby("year_month")[["no2_exceed","pm25_exceed"]].sum().reset_index()
exc_monthly["ym_str"] = exc_monthly["year_month"].astype(str)

fig, ax = plt.subplots(figsize=(14, 4))
x = np.arange(len(exc_monthly))
w = 0.38
ax.bar(x - w/2, exc_monthly["no2_exceed"],  w, label=f"NO₂ > {WHO['no2']} µg/m³",   color="#1a6bbf", alpha=0.85)
ax.bar(x + w/2, exc_monthly["pm25_exceed"], w, label=f"PM2.5 > {WHO['pm2p5']} µg/m³", color="#e07b39", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(exc_monthly["ym_str"], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Days exceeding WHO guideline")
ax.set_title("Monthly WHO 24-h guideline exceedances — Paris, 2022–2023", fontsize=13, pad=10)
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## 3 — Calendar heatmap (NO₂)

Each cell = one day; colour intensity = NO₂ concentration.  
Red outline marks days above the WHO 24-h guideline.

In [ ]:
def calendar_heatmap(series: pd.Series, title: str, threshold: float, cmap="YlOrRd"):
    """Draw a week × week-of-year heatmap for a daily Series indexed by date."""
    s = series.copy()
    s.index = pd.to_datetime(s.index)
    years = sorted(s.index.year.unique())

    fig, axes = plt.subplots(1, len(years), figsize=(14, 3 * len(years)), sharey=True)
    if len(years) == 1:
        axes = [axes]

    vmin, vmax = s.min(), max(s.max(), threshold * 1.1)
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    mapper = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

    for ax, year in zip(axes, years):
        yr = s[s.index.year == year]
        # pivot: rows = day-of-week (0=Mon), cols = week-of-year
        pivot = pd.DataFrame({"val": yr.values, "dow": yr.index.dayofweek,
                               "woy": yr.index.isocalendar().week.values})
        grid = pivot.pivot_table(index="dow", columns="woy", values="val")

        im = ax.imshow(grid.values, aspect="auto", cmap=cmap, norm=norm)

        # red border on exceedance cells
        for (r, c), val in np.ndenumerate(grid.values):
            if not np.isnan(val) and val > threshold:
                ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                           fill=False, edgecolor="#c0392b", linewidth=1.3))

        ax.set_yticks(range(7))
        ax.set_yticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"], fontsize=8)
        ax.set_xlabel("Week of year")
        ax.set_title(str(year), fontsize=11)

    fig.suptitle(title, fontsize=13, y=1.01)
    fig.colorbar(mapper, ax=axes, orientation="vertical", fraction=0.02, pad=0.04,
                 label="µg/m³")
    fig.tight_layout()
    plt.show()

no2_daily = daily.set_index("date")["no2_mean"]
calendar_heatmap(no2_daily, "Daily NO₂ — Paris (red = WHO exceedance)", WHO["no2"])

## 4 — Use the built-in exceedance API

The API can also compute exceedance hours server-side (cheaper — single request).

In [ ]:
exc = client.query(
    lat=(48.7, 49.0),
    lon=(2.2, 2.5),
    start="2023-01",
    end="2023-12",
    variables=["no2"],
    aggregate="exceedance",
    threshold=WHO["no2"],
)

print(exc.head(10).to_string(index=False))
print(f"\nGrid points with >0 exceedance hours: {(exc['hours_above'] > 0).sum()}")

---
**Next:** See [04_multi_city_comparison.ipynb](04_multi_city_comparison.ipynb) to compare air quality across European cities.